In [1]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

SOLUTION_DIR = (
    Path.home()
    / "Documents/GitHub/ml-zoomcamp-2026-homework"
    / "05-deploying"
)

assert SOLUTION_DIR.is_dir(), (
    f"Solution directory not found: {SOLUTION_DIR}"
)

REFERENCE_DIR = SOLUTION_DIR / "reference"
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

COMMIT = "9bb19da5777191f0dc0a038f93ef1b429db5a2bb"

BASE_URL = (
    "https://raw.githubusercontent.com/"
    f"DataTalksClub/machine-learning-zoomcamp/{COMMIT}/"
    "cohorts/2026/homework/05-deployment"
)

FILES = [
    ".python-version",
    "Dockerfile",
    "README.md",
    "feature_defaults.json",
    "homework.md",
    "homework.yaml",
    "model.py",
    "model_metadata.json",
    "pipeline.bin",
    "predict.py",
    "pyproject.toml",
    "q6_test.py",
    "smoke_test.py",
    "train.py",
    "uv.lock",
]

for filename in FILES:
    url = f"{BASE_URL}/{filename}"

    with urlopen(url, timeout=30) as response:
        content = response.read()

    destination = REFERENCE_DIR / filename
    destination.write_bytes(content)

    print(f"Downloaded {filename}: {len(content):,} bytes")

expected_checksum = (
    "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
)

actual_checksum = hashlib.sha256(
    (REFERENCE_DIR / "pipeline.bin").read_bytes()
).hexdigest()

print("\nModel checksum:", actual_checksum)
assert actual_checksum == expected_checksum, (
    "The model checksum does not match the homework instructions."
)

print("All reference files downloaded and model checksum verified.")

Downloaded .python-version: 8 bytes
Downloaded Dockerfile: 598 bytes
Downloaded README.md: 1,044 bytes
Downloaded feature_defaults.json: 116 bytes
Downloaded homework.md: 6,077 bytes
Downloaded homework.yaml: 563 bytes
Downloaded model.py: 1,456 bytes
Downloaded model_metadata.json: 872 bytes
Downloaded pipeline.bin: 1,941 bytes
Downloaded predict.py: 1,426 bytes
Downloaded pyproject.toml: 326 bytes
Downloaded q6_test.py: 828 bytes
Downloaded smoke_test.py: 777 bytes
Downloaded train.py: 3,311 bytes
Downloaded uv.lock: 32,279 bytes

Model checksum: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
All reference files downloaded and model checksum verified.


In [2]:
from pathlib import Path
import subprocess

SOLUTION_DIR = (
    Path.home()
    / "Documents/GitHub/ml-zoomcamp-2026-homework"
    / "05-deploying"
)

REFERENCE_DIR = SOLUTION_DIR / "reference"

UV = Path.home() / ".var/app/com.visualstudio.code/bin/uv"

assert UV.is_file(), f"uv executable not found: {UV}"
assert (REFERENCE_DIR / "pyproject.toml").is_file()
assert (REFERENCE_DIR / "uv.lock").is_file()

print("Reference directory:", REFERENCE_DIR)

subprocess.run(
    [str(UV), "sync", "--locked"],
    cwd=REFERENCE_DIR,
    check=True,
)

Reference directory: /home/maometto/Documents/GitHub/ml-zoomcamp-2026-homework/05-deploying/reference


 Downloaded cpython-3.11.15-linux-x86_64-gnu (download)
Using CPython 3.11.15
Creating virtual environment at: .venv
Resolved 19 packages in 0.92ms
 Downloaded pydantic-core
 Downloaded scikit-learn
 Downloaded numpy
 Downloaded scipy
Prepared 18 packages in 46.94s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 18 packages in 761ms
 + annotated-types==0.8.0
 + anyio==4.15.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + fastapi==0.119.1
 + h11==0.16.0
 + idna==3.19
 + joblib==1.6.0
 + numpy==2.3.3
 + pydantic==2.13.5
 + pydantic-core==2.46.5
 + scikit-learn==1.7.2
 + scipy==1.17.1
 + starlette==0.48.0
 + threadpoolctl==3.6.0
 + typing-extensions==4.16.0
 + typing-inspection==0.4.4
 + uvicorn==0.38.0


CompletedProcess(args=['/home/maometto/.var/app/com.visualstudio.code/bin/uv', 'sync', '--locked'], returncode=0)

In [3]:
import subprocess

subprocess.run(
    [str(UV), "run", "python", "smoke_test.py"],
    cwd=REFERENCE_DIR,
    check=True,
)

{'conversion_probability': 0.769799, 'conversion': True}


CompletedProcess(args=['/home/maometto/.var/app/com.visualstudio.code/bin/uv', 'run', 'python', 'smoke_test.py'], returncode=0)

In [4]:
import hashlib

model_path = REFERENCE_DIR / "pipeline.bin"

with open(model_path, "rb") as f:
    actual_checksum = hashlib.sha256(f.read()).hexdigest()

expected_checksum = (
    "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
)

print("Actual checksum:  ", actual_checksum)
print("Expected checksum:", expected_checksum)
print("Matches:", actual_checksum == expected_checksum)

assert actual_checksum == expected_checksum, "Model checksum mismatch!"

Actual checksum:   1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Expected checksum: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Matches: True


### Question 1

In [5]:
import subprocess

result = subprocess.run(
    [str(UV), "--version"],
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout.strip())

uv 0.13.5 (x86_64-unknown-linux-gnu)


### Question 2

In [6]:
import tomllib

lock_path = REFERENCE_DIR / "uv.lock"

with open(lock_path, "rb") as f:
    lock_data = tomllib.load(f)

sklearn_versions = [
    package["version"]
    for package in lock_data["package"]
    if package["name"] == "scikit-learn"
]

print("Scikit-Learn version in uv.lock:", sklearn_versions)

Scikit-Learn version in uv.lock: ['1.7.2']


### Question 3

In [8]:
print((REFERENCE_DIR / "smoke_test.py").read_text())

from __future__ import annotations

import pickle
from pathlib import Path

from model import normalize_record


MODEL_PATH = Path(__file__).with_name("pipeline.bin")
CLIENT = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

with MODEL_PATH.open("rb") as model_file:
    pipeline = pickle.load(model_file)

record = normalize_record(CLIENT)
first = float(pipeline.predict_proba([record])[0, 1])
second = float(pipeline.predict_proba([record])[0, 1])

assert 0 < first < 1
assert first == second
print({"conversion_probability": round(first, 6), "conversion": first >= 0.5})



In [9]:
q3_script = """
import pickle
from model import normalize_record

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

with open("pipeline.bin", "rb") as f:
    pipeline = pickle.load(f)

record = normalize_record(lead)
probability = float(pipeline.predict_proba([record])[0, 1])

print(f"Unrounded probability: {probability:.10f}")
print(f"Q3 answer: {probability:.3f}")
"""

result = subprocess.run(
    [
        str(UV), "run", "--project", str(REFERENCE_DIR),
        "python", "-c", q3_script,
    ],
    cwd=REFERENCE_DIR,
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout)

Unrounded probability: 0.5329944186
Q3 answer: 0.533



### Question 4

I started the provided FastAPI application with Uvicorn and sent
the second lead to the `POST /predict` endpoint.

I checked that the response contains a conversion probability
between 0 and 1 and a boolean conversion prediction.

In [ ]:
import json
from urllib.request import Request, urlopen

API_URL = "http://127.0.0.1:9696"

with urlopen(f"{API_URL}/health", timeout=10) as response:
    health = json.load(response)

print("Health:", health)

lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

request = Request(
    f"{API_URL}/predict",
    data=json.dumps(lead_q4).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urlopen(request, timeout=10) as response:
    result_q4 = json.load(response)

print("Prediction:", result_q4)

probability_q4 = float(result_q4["conversion_probability"])
conversion_q4 = result_q4["conversion"]

assert 0 <= probability_q4 <= 1
assert isinstance(conversion_q4, bool)

print(f"Q4 answer: {probability_q4:.3f}")

Health: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
Prediction: {'conversion_probability': 0.769799, 'conversion': True}
Q4 answer: 0.770


### Question 5

In [11]:
dockerfile_path = REFERENCE_DIR / "Dockerfile"

for line in dockerfile_path.read_text().splitlines():
    if line.strip().upper().startswith("FROM "):
        print(line)

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3


### Question 6

I built and ran the Docker image using the provided Dockerfile.
I tested the prediction using `q6_test.py`.

The container returned a conversion probability of **0.770**
(rounded to three decimal places), with `conversion: True`.

The prediction matches Question 4 exactly, confirming that the
container produces the same inference as the directly served API.

In [14]:
import json
from urllib.request import urlopen

expected_model_sha256 = (
    "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
)

with urlopen("http://localhost:9696/health", timeout=10) as response:
    health = json.load(response)

print(json.dumps(health, indent=2))

assert health["status"] == "ok"
assert health["model_sha256"] == expected_model_sha256

print("Health check passed.")

{
  "status": "ok",
  "model_sha256": "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1",
  "feature_defaults_sha256": "1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"
}
Health check passed.


In [15]:
result = subprocess.run(
    [str(UV), "run", "python", "q6_test.py"],
    cwd=REFERENCE_DIR,
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout)

{'conversion_probability': 0.769799, 'conversion': True}

